# Know Your Batik — Data Preprocessing

In [ ]:
import hashlib
import shutil
import sys
import warnings
from pathlib import Path

import pandas as pd
import yaml
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# resolve project root (notebooks/ is one level below root)
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

with open(PROJECT_ROOT / 'config.yaml') as f:
    CFG = yaml.safe_load(f)

DATASET_DIR   = Path(r'C:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\Know Your Batik\dataset')
PROCESSED_DIR = PROJECT_ROOT / CFG['data']['processed_path']
VALID_EXTS    = {'.jpg', '.jpeg', '.png', '.webp'}

TRAIN_RATIO = CFG['data']['split']['train']
VAL_RATIO   = CFG['data']['split']['val']
TEST_RATIO  = CFG['data']['split']['test']

## Step 1 — LOAD CLEAN DF

In [ ]:
# ── build raw dataframe ──────────────────────────────────────────────────────
records = []
for cls_dir in sorted(DATASET_DIR.iterdir()):
    if not cls_dir.is_dir():
        continue
    for img_path in cls_dir.iterdir():
        if img_path.suffix.lower() in VALID_EXTS:
            records.append({
                'path':       str(img_path),
                'class_name': cls_dir.name,
                'filename':   img_path.name,
                'extension':  img_path.suffix.lower(),
            })
df = pd.DataFrame(records)

# ── PIL verify → remove corrupt ──────────────────────────────────────────────
def is_corrupt(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return False
    except Exception:
        return True

corrupt_mask = df['path'].apply(is_corrupt)
df = df[~corrupt_mask].reset_index(drop=True)
print(f'Corrupt removed  : {corrupt_mask.sum()}')

# ── MD5 dedup → keep first ───────────────────────────────────────────────────
def md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

df['md5'] = df['path'].apply(md5)
dup_mask  = df.duplicated(subset='md5', keep='first')
df        = df[~dup_mask].reset_index(drop=True)
print(f'Duplicates removed : {dup_mask.sum()}')
print(f'Final clean df size: {len(df)} images across {df["class_name"].nunique()} classes')

## Step 2 — STRATIFIED SPLIT

In [ ]:
# train = 80 %, then split remainder 50/50 for val/test (→ 10/10 overall)
val_test_ratio = VAL_RATIO / (VAL_RATIO + TEST_RATIO)

df_train, df_temp = train_test_split(
    df, test_size=(1 - TRAIN_RATIO),
    stratify=df['class_name'], random_state=42,
)
df_val, df_test = train_test_split(
    df_temp, test_size=(1 - val_test_ratio),
    stratify=df_temp['class_name'], random_state=42,
)

splits = {'train': df_train, 'val': df_val, 'test': df_test}

print(f'Train : {len(df_train):>5}  ({len(df_train)/len(df)*100:.1f}%)')
print(f'Val   : {len(df_val):>5}  ({len(df_val)/len(df)*100:.1f}%)')
print(f'Test  : {len(df_test):>5}  ({len(df_test)/len(df)*100:.1f}%)')
print()

# per-class distribution check
dist = pd.DataFrame({
    'train': df_train['class_name'].value_counts(),
    'val':   df_val['class_name'].value_counts(),
    'test':  df_test['class_name'].value_counts(),
}).sort_index()
print(dist.to_string())

## Step 3 — COPY TO PROCESSED FOLDERS

In [ ]:
for split_name, split_df in splits.items():
    copied = 0
    for _, row in split_df.iterrows():
        dest_dir = PROCESSED_DIR / split_name / row['class_name']
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest_file = dest_dir / row['filename']
        if not dest_file.exists():
            shutil.copy2(row['path'], dest_file)
        copied += 1
    print(f'{split_name:>5} — copied {copied} images')

## Step 4 — VERIFY

In [ ]:
rows = []
total_found = 0

all_classes = sorted(df['class_name'].unique())
for cls in all_classes:
    row = {'class': cls}
    for split in ('train', 'val', 'test'):
        cls_dir = PROCESSED_DIR / split / cls
        n = len(list(cls_dir.glob('*'))) if cls_dir.exists() else 0
        row[split] = n
        total_found += n
    rows.append(row)

verify_df = pd.DataFrame(rows).set_index('class')
verify_df['total'] = verify_df.sum(axis=1)

print(verify_df.to_string())
print(f'\nGrand total in processed/ : {total_found}')
print(f'Expected (clean df)       : {len(df)}')

assert total_found == len(df), (
    f'MISMATCH: found {total_found}, expected {len(df)}'
)
print('Assertion passed — all images accounted for.')

## Step 5 — PREPROCESSING PIPELINE (src/preprocessor.py)

Already written to `src/preprocessor.py`. Displaying the source for reference:

In [ ]:
print((PROJECT_ROOT / 'src' / 'preprocessor.py').read_text())

## Step 6 — DATALOADER (src/data_loader.py)

Already written to `src/data_loader.py`. Displaying the source for reference:

In [ ]:
print((PROJECT_ROOT / 'src' / 'data_loader.py').read_text())

## Step 7 — SMOKE TEST

In [ ]:
import torch
from src.data_loader import get_dataloaders

train_loader, val_loader, test_loader = get_dataloaders(
    processed_path=PROCESSED_DIR,
    batch_size=4,
    num_workers=0,  # 0 avoids multiprocessing issues inside notebooks
)

images, labels = next(iter(train_loader))

print(f'Batch shape  : {images.shape}')
print(f'Label tensor : {labels}')
print(f'Pixel min    : {images.min():.4f}')
print(f'Pixel max    : {images.max():.4f}')

expected = torch.Size([4, 3, 224, 224])
assert images.shape == expected, f'Shape mismatch: {images.shape} != {expected}'
print(f'\nShape assertion passed — {images.shape}')